In [ ]:

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

print("Library berhasil dipanggil!")
print("Tanggal:", datetime.now())


Library berhasil dipanggil!
Tanggal: 2026-11-06 11:31:48.988534


In [9]:
import requests
import json
import time
from datetime import datetime
from pathlib import Path

URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": "INSTIKI-BIFP243-Praktikum/1.0 (nim@instiki.ac.id)"}

# Query Overpass untuk mengambil semua akomodasi pariwisata di area Kintamani
QUERY = """
[out:json][timeout:90];
area["name"="Kintamani"]->.searchArea;
(
  node["tourism"~"hotel|guest_house|hostel|motel|resort|apartment"](area.searchArea);
  way["tourism"~"hotel|guest_house|hostel|motel|resort|apartment"](area.searchArea);
  relation["tourism"~"hotel|guest_house|hostel|motel|resort|apartment"](area.searchArea);
);
out body;
>;
out skel qt;
"""

def ambil_overpass(query, maks_percobaan=4, jeda=5):
    for percobaan in range(1, maks_percobaan + 1):
        try:
            r = requests.post(URL, data={"data": query}, headers=HEADERS, timeout=120)
            if r.status_code == 200:
                return r.json()
            print(f"[percobaan {percobaan}] status {r.status_code}, tunggu {jeda}s")
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[percobaan {percobaan}] {type(e).__name__}, tunggu {jeda}s")

        time.sleep(jeda)
        jeda *= 2
    raise RuntimeError("Gagal mengambil data dari Overpass")

data = ambil_overpass(QUERY)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
Path("data/raw").mkdir(parents=True, exist_ok=True)
raw_path = Path(f"data/raw/osm_akomodasi_kintamani_{stamp}.json")
raw_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Jumlah elemen:", len(data["elements"]))


[percobaan 1] status 502, tunggu 5s
Jumlah elemen: 34


In [12]:
import hashlib
from datetime import datetime, timezone
from pathlib import Path
import json

nama_pengambil = "Angelina Florensiana Jelita"
nim_pengambil = "2501010371"

waktu_akuisisi = datetime.now(timezone.utc).isoformat()

hash_md5 = hashlib.md5(raw_path.read_bytes()).hexdigest()

provenance = {
    "url_endpoint": URL,
    "query": QUERY,
    "waktu_akuisisi_utc": waktu_akuisisi,
    "jumlah_elemen": len(data["elements"]),
    "nama_file_mentah": raw_path.name,
    "hash_md5": hash_md5,
    "lisensi": "Open Database License (ODbL)",
    "atribusi": "© OpenStreetMap contributors",
    "nama_pengambil": nama_pengambil,
    "nim_pengambil": nim_pengambil
}

Path("docs").mkdir(parents=True, exist_ok=True)

provenance_path = Path("docs/provenance.json")

provenance_path.write_text(
    json.dumps(provenance, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Provenance berhasil dibuat!")
print("Lokasi:", provenance_path)


Provenance berhasil dibuat!
Lokasi: docs/provenance.json


In [15]:
def ratakan(elements):
    baris = []

    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)

        baris.append({
            "osm_type": e["type"],
            "osm_id": e["id"],
            "nama": tg.get("name"),
            "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"),
            "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"),
            "bintang": tg.get("stars"),
            "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}",
        })

    return pd.DataFrame(baris)

In [17]:
import pandas as pd

akomodasi = ratakan(data["elements"])

print(akomodasi.head())


  osm_type      osm_id                     nama        jenis       lat  \
0     node  1977779907  The Lakeview Hotel Bali        hotel -8.283140   
1     node  2107226701                   Arlina  guest_house -8.249815   
2     node  2107227001      Toya Devasya Resort        hotel -8.250164   
3     node  4150326329               Guesthouse        hotel -8.281456   
4     node  4297310696           The Cave Hotel        hotel -8.280748   

          lon  desa bintang jumlah_kamar                      website  \
0  115.365844  None    None         None  http://lakeviewbali.com/en/   
1  115.398465  None    None         None                         None   
2  115.399798  None    None         None                         None   
3  115.362223  None    None         None                         None   
4  115.362048  None       2            5                         None   

                                         url_osm  
0  https://www.openstreetmap.org/node/1977779907  
1  https://www

In [18]:
print(akomodasi["jenis"].value_counts())

jenis
hotel          16
guest_house     9
hostel          4
motel           1
Name: count, dtype: int64


In [20]:
print("Tanpa nama:", akomodasi["nama"].isna().sum())

Tanpa nama: 5


In [21]:
duplikat = akomodasi.duplicated(
    subset=["osm_type", "osm_id"]
).sum()

print("Jumlah data duplikat:", duplikat)

Jumlah data duplikat: 0


In [22]:
tanpa_nama = akomodasi[akomodasi["nama"].isna()]

print("Jumlah akomodasi tanpa nama:", len(tanpa_nama))
display(tanpa_nama)

Jumlah akomodasi tanpa nama: 5


,osm_type,osm_id,nama,jenis,lat,lon,desa,bintang,jumlah_kamar,website,url_osm
29,way,1181397894,None,motel,NaN,NaN,None,None,None,None,https://www.openstreetmap.org/way/1181397894
30,node,10971340490,None,None,-8.223228,115.254525,None,None,None,None,https://www.openstreetmap.org/node/10971340490
31,node,10971340491,None,None,-8.223525,115.254566,None,None,None,None,https://www.openstreetmap.org/node/10971340491
32,node,10971340492,None,None,-8.223515,115.254639,None,None,None,None,https://www.openstreetmap.org/node/10971340492
33,node,10971340493,None,None,-8.223218,115.254598,None,None,None,None,https://www.openstreetmap.org/node/10971340493


In [23]:
print("Koordinat latitude kosong:",
      akomodasi["lat"].isna().sum())

print("Koordinat longitude kosong:",
      akomodasi["lon"].isna().sum())

print("Rentang latitude:")
print(akomodasi["lat"].min(), "-", akomodasi["lat"].max())

print("Rentang longitude:")
print(akomodasi["lon"].min(), "-", akomodasi["lon"].max())

Koordinat latitude kosong: 1
Koordinat longitude kosong: 1
Rentang latitude:
-8.2837958 - -8.2189435
Rentang longitude:
115.2545251 - 115.4129261


In [24]:
persentase_kosong = (
    akomodasi.isna().mean() * 100
).round(2)

print("Persentase data kosong setiap kolom:")
print(persentase_kosong)

Persentase data kosong setiap kolom:
osm_type          0.00
osm_id            0.00
nama             14.71
jenis            11.76
lat               2.94
lon               2.94
desa            100.00
bintang          97.06
jumlah_kamar     94.12
website          94.12
url_osm           0.00
dtype: float64


In [25]:
Path("data/interim").mkdir(parents=True, exist_ok=True)

csv_path = Path("data/interim/akomodasi_kintamani.csv")

akomodasi.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("Dataset CSV berhasil disimpan!")
print("Lokasi:", csv_path)
print("Jumlah baris:", len(akomodasi))
print("Jumlah kolom:", len(akomodasi.columns))

Dataset CSV berhasil disimpan!
Lokasi: data/interim/akomodasi_kintamani.csv
Jumlah baris: 34
Jumlah kolom: 11


In [29]:
!pip install shapely

In [30]:
from shapely.geometry import Point, Polygon
from shapely.ops import unary_union, polygonize

print("Library pemeriksaan koordinat berhasil dipanggil!")

Library pemeriksaan koordinat berhasil dipanggil!


In [31]:
QUERY_BATAS = """
[out:json][timeout:90];
relation["boundary"="administrative"]
["name"="Kintamani"]
["admin_level"="6"];
out tags;
"""

batas_data = ambil_overpass(QUERY_BATAS)

print("Jumlah batas wilayah ditemukan:",
      len(batas_data["elements"]))

for e in batas_data["elements"]:
    print("ID:", e["id"])
    print("Nama:", e["tags"].get("name"))
    print("Admin level:", e["tags"].get("admin_level"))

Jumlah batas wilayah ditemukan: 1
ID: 20447233
Nama: Kintamani
Admin level: 6


In [32]:
QUERY_GEOMETRI = """
[out:json][timeout:90];
relation(20447233);
out geom;
"""

batas_geom_data = ambil_overpass(QUERY_GEOMETRI)

print("Geometri batas wilayah berhasil diambil!")
print("Jumlah elemen:",
      len(batas_geom_data["elements"]))

print("Jumlah anggota relasi:",
      len(batas_geom_data["elements"][0]["members"]))

Geometri batas wilayah berhasil diambil!
Jumlah elemen: 1
Jumlah anggota relasi: 57


In [33]:
# Jumlah akomodasi dari OpenStreetMap
jumlah_osm = len(akomodasi)

# Jumlah akomodasi berdasarkan data BPS tahun 2023
jumlah_hotel_bps = 19
jumlah_penginapan_bps = 99

jumlah_bps = jumlah_hotel_bps + jumlah_penginapan_bps

# Menghitung persentase kelengkapan
persentase = (jumlah_osm / jumlah_bps) * 100

print("HASIL UJI KELENGKAPAN DATA")
print("Jumlah OSM:", jumlah_osm)
print("Jumlah BPS:", jumlah_bps)
print("Tahun BPS: 2023")
print("Persentase kelengkapan:", round(persentase, 2), "%")

HASIL UJI KELENGKAPAN DATA
Jumlah OSM: 34
Jumlah BPS: 118
Tahun BPS: 2023
Persentase kelengkapan: 28.81 %


In [34]:
# Memilih 10 data secara acak
sampel_10 = akomodasi.sample(
    n=10,
    random_state=42
)

# Menampilkan data yang terpilih
display(
    sampel_10[
        ["osm_type", "osm_id", "nama", "jenis",
         "lat", "lon", "url_osm"]
    ]
)

,osm_type,osm_id,nama,jenis,lat,lon,url_osm
15,node,7058054537,Segara Hotel,hotel,-8.278086,115.379134,https://www.openstreetmap.org/node/7058054537
19,node,7238422213,Batur Panorama,hotel,-8.218943,115.392130,https://www.openstreetmap.org/node/7238422213
27,node,13068182153,Batur Sunrise Guesthouse,hotel,-8.249020,115.400863,https://www.openstreetmap.org/node/13068182153
26,node,12873439817,Pandawa Glamping Kintamani,hotel,-8.235285,115.407874,https://www.openstreetmap.org/node/12873439817
8,node,5065920543,The Volcania Guesthouse,hotel,-8.283796,115.365242,https://www.openstreetmap.org/node/5065920543
24,node,10983432080,Batur View Homestay,guest_house,-8.279777,115.357832,https://www.openstreetmap.org/node/10983432080
21,node,9555392317,Pondok Bali Cottage,guest_house,-8.222104,115.398161,https://www.openstreetmap.org/node/9555392317
12,node,5668936157,Black Lava Hostel,hostel,-8.246333,115.396276,https://www.openstreetmap.org/node/5668936157
32,node,10971340492,None,None,-8.223515,115.254639,https://www.openstreetmap.org/node/10971340492
9,node,5175562421,Entrance to: Bunbulan Hill Hostel,hostel,-8.222014,115.395349,https://www.openstreetmap.org/node/5175562421


In [35]:
hasil_verifikasi = sampel_10[
    ["osm_type", "osm_id", "nama", "jenis",
     "lat", "lon", "url_osm"]
].copy()

hasil_verifikasi["verifikasi_nama"] = ""
hasil_verifikasi["verifikasi_jenis"] = ""
hasil_verifikasi["verifikasi_lokasi"] = ""
hasil_verifikasi["sumber_verifikasi"] = ""
hasil_verifikasi["catatan"] = ""

display(hasil_verifikasi)

,osm_type,osm_id,nama,jenis,lat,lon,url_osm,verifikasi_nama,verifikasi_jenis,verifikasi_lokasi,sumber_verifikasi,catatan
15,node,7058054537,Segara Hotel,hotel,-8.278086,115.379134,https://www.openstreetmap.org/node/7058054537,,,,,
19,node,7238422213,Batur Panorama,hotel,-8.218943,115.392130,https://www.openstreetmap.org/node/7238422213,,,,,
27,node,13068182153,Batur Sunrise Guesthouse,hotel,-8.249020,115.400863,https://www.openstreetmap.org/node/13068182153,,,,,
26,node,12873439817,Pandawa Glamping Kintamani,hotel,-8.235285,115.407874,https://www.openstreetmap.org/node/12873439817,,,,,
8,node,5065920543,The Volcania Guesthouse,hotel,-8.283796,115.365242,https://www.openstreetmap.org/node/5065920543,,,,,
24,node,10983432080,Batur View Homestay,guest_house,-8.279777,115.357832,https://www.openstreetmap.org/node/10983432080,,,,,
21,node,9555392317,Pondok Bali Cottage,guest_house,-8.222104,115.398161,https://www.openstreetmap.org/node/9555392317,,,,,
12,node,5668936157,Black Lava Hostel,hostel,-8.246333,115.396276,https://www.openstreetmap.org/node/5668936157,,,,,
32,node,10971340492,None,None,-8.223515,115.254639,https://www.openstreetmap.org/node/10971340492,,,,,
9,node,5175562421,Entrance to: Bunbulan Hill Hostel,hostel,-8.222014,115.395349,https://www.openstreetmap.org/node/5175562421,,,,,


In [41]:
from pathlib import Path

Path("docs").mkdir(exist_ok=True)

hasil_verifikasi.to_csv(
    "docs/verifikasi_10_sampel.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File verifikasi berhasil disimpan!")

File verifikasi berhasil disimpan!


In [42]:
print("Jumlah sampel:", len(hasil_verifikasi))

display(hasil_verifikasi)

Jumlah sampel: 10


,osm_type,osm_id,nama,jenis,lat,lon,url_osm,verifikasi_nama,verifikasi_jenis,verifikasi_lokasi,sumber_verifikasi,catatan
15,node,7058054537,Segara Hotel,hotel,-8.278086,115.379134,https://www.openstreetmap.org/node/7058054537,,,,,
19,node,7238422213,Batur Panorama,hotel,-8.218943,115.392130,https://www.openstreetmap.org/node/7238422213,,,,,
27,node,13068182153,Batur Sunrise Guesthouse,hotel,-8.249020,115.400863,https://www.openstreetmap.org/node/13068182153,,,,,
26,node,12873439817,Pandawa Glamping Kintamani,hotel,-8.235285,115.407874,https://www.openstreetmap.org/node/12873439817,,,,,
8,node,5065920543,The Volcania Guesthouse,hotel,-8.283796,115.365242,https://www.openstreetmap.org/node/5065920543,,,,,
24,node,10983432080,Batur View Homestay,guest_house,-8.279777,115.357832,https://www.openstreetmap.org/node/10983432080,,,,,
21,node,9555392317,Pondok Bali Cottage,guest_house,-8.222104,115.398161,https://www.openstreetmap.org/node/9555392317,,,,,
12,node,5668936157,Black Lava Hostel,hostel,-8.246333,115.396276,https://www.openstreetmap.org/node/5668936157,,,,,
32,node,10971340492,None,None,-8.223515,115.254639,https://www.openstreetmap.org/node/10971340492,,,,,
9,node,5175562421,Entrance to: Bunbulan Hill Hostel,hostel,-8.222014,115.395349,https://www.openstreetmap.org/node/5175562421,,,,,
